# INF201 - Deliverable 4 
## *Temperatures and Fractions*

**Group:** G6_2: Nikolai Friis, Tommy Rustad Apelseth, Aron Lerdahl

**GitHub repository:** https://github.com/tommyra06/INF201-Deliverables

---

### Contents
* Setup
* Task 1: *Temperatures*
* Task 2: *Fractions*
* Use of generative AI

---

## Setup

### Virtual environment

Create a new virtual environment (`.venv`). Install the packages for Deliverable 4, listed in `requirements.txt`. Use the command palette.

### Python scripts
The classes are written in their own scripts so the tests can import them with `pytest`:

* `temperature_reader.py` - *the `Temperature` class (Task 1)*
* `test_temperature.py` - *pytest tests for the `Temperature` class (Task 1)*
* `fraction.py` - *the `Fraction` class (Task 2)*
* `test_fraction.py` - *pytest tests for the `Fraction` class (Task 2)*

### Imports
All imports used in the notebook are collected in the cell below.

In [2]:
# Library imports that come with Python
import math  # Used for math.gcd to reduce fractions to lowest terms (Task 2)

# Our own classes, from the scripts in this folder
# from temperature_reader import Temperature  # Task 1
from fraction import Fraction  # Task 2

---

## Task 1: *Temperatures*

For Task 1, we have created the `Temperature` class that supports Celsius, Fahrenheit and Kelvin.

* `to_celsius`, `to_fahrenheit`, `to_kelvin` converts temperature to C, F or K
* `convert_to` activates the `to_XXXX` methods based of the target unit
* `__repr__`, `__str__` generates string representations of the class
* `__eq__`, `__ne__`, `__lt__`, `__le__`, `__gt__`, `__ge__`,  methods to compare with another object

In [3]:
class Temperature:
    """Class that represents temperature with unit"""

    def __init__(self, value: float, unit: str):
        """
                Initialise class
        
                Args:
                    value (float): The temperature value
                    unit (str): The temperature unit
                """
        unit = unit.upper()
        absolue_zero = {
            'K': 0, 
            'C': -273.15, 
            'F': -459.67
            }
        
        if unit not in absolue_zero or value < self.absolue_zero[unit] - 1e-9:
            raise ValueError('Invalid unit')
        #Checks if given unit is valid and if given value is above absolute zero
        
        self._value = value
        self._unit = unit

    def to_celsius(self) -> float:
        """Converts temperature to celsius"""
        if self._unit == 'F':
            return((self._value - 32) * (5 / 9))
        elif self._unit == 'K':
            return(self._value - 273.15)
        else:
            return(self._value)

    def to_fahrenheit(self) -> float:
        """Converts temperature to fahrenheit"""
        if self._unit == 'C':
            return((self._value * (9 / 5)) + 32)
        elif self._unit == 'K':
            return((self._value - 273.15) * (9 / 5) + 32)
        else:
            return(self._value)

    def to_kelvin(self) -> float:
        """Converts temperature to kelvin"""
        if self._unit == 'C':
            return(self._value + 273.15)
        elif self._unit == 'F':
            return((self._value - 32) * (5 / 9) + 273.15)
        else:
            return(self._value)

    def convert_to(self, target_unit: str) -> object:
        """Creates new object with converted temperature to given unit"""
        converters = {
            'K': self.to_kelvin, 
            'F': self.to_fahrenheit, 
            'C': self.to_celsius
            }
        target_unit = target_unit.upper()

        if target_unit not in converters:
            raise ValueError('Invalid unit')
        
        return Temperature(converters[target_unit](), target_unit)

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Temperature):
            return NotImplemented
        return abs(self.to_kelvin() - other.to_kelvin()) <= 1e-9

    def __ne__(self, other: object) -> bool:
        if not isinstance(other, Temperature):
            return NotImplemented
        return not self.__eq__(other)

    def __lt__(self, other: object) -> bool:
        if not isinstance(other, Temperature):
            return NotImplemented
        return self.to_kelvin() < other.to_kelvin()

    def __le__(self, other: object) -> bool:
        if not isinstance(other, Temperature):
            return NotImplemented
        return self.to_kelvin() <= other.to_kelvin()

    def __gt__(self, other: object) -> bool:
        if not isinstance(other, Temperature):
            return NotImplemented
        return other < self

    def __ge__(self, other: object) -> bool:
        if not isinstance(other, Temperature):
            return NotImplemented
        return other <= self
    
    def __repr__(self) -> str:
        return(f'Temperature({self._value}, {self._unit})')

    def __str__(self) -> str:
        degree_sign = '\N{DEGREE SIGN}'
        return(f'{self._value:.2f} {degree_sign + self._unit}')

    @property
    def value(self) -> float:
        return self._value

    @property
    def unit(self) -> str:
        return self._unit



        

---

### Testing with pytest

Tests for a lot of things
1. If class was initialized correctly
2. If attributes are read_only
3. If given unit and temperature are valid
4. Different conversions
5. Roundtrip of all conversions
6. All comparisons
7. if str and repr methods are correctly formatted

In [4]:
!pytest -v test_temperature.py

def test_initialization():
    temp = Temperature(25, 'C')
    assert temp.value == 25
    assert temp.unit == 'C'

def test_read_only_properties():
    temp = Temperature(25, 'C')

    with pytest.raises(AttributeError):
        temp.value = 30

    with pytest.raises(AttributeError):
        temp.unit = 'F'

def test_validation():
    with pytest.raises(ValueError):
        Temperature(10, 'W')

    with pytest.raises(ValueError):
        Temperature(-1, 'K')

    with pytest.raises(ValueError):
        Temperature(-274, 'C')

    with pytest.raises(ValueError):
        Temperature(-460, 'F')

def test_conversions():
    assert Temperature(-273.15, 'C').to_fahrenheit() == pytest.approx(-459.67)
    assert Temperature(-459.67, 'F').to_kelvin() == pytest.approx(0)

    assert Temperature(-40, 'F').to_kelvin() == pytest.approx(233.15)
    assert Temperature(233.15, 'K').to_celsius() == pytest.approx(-40)

    assert Temperature(273.15, 'K').to_fahrenheit() == pytest.approx(32)
    assert Temperature(32, 'F').to_celsius() == pytest.approx(0)

    assert Temperature(37, 'C').to_kelvin() == pytest.approx(310.15)
    assert Temperature(310.15, 'K').to_fahrenheit() == pytest.approx(98.6)

    assert Temperature(100, 'C').to_fahrenheit() == pytest.approx(212)
    assert Temperature(212, 'F').to_kelvin() == pytest.approx(373.15)

def test_roundtrip():
    temp = Temperature(10, 'C')
    converted = temp.convert_to('F')
    converted = temp.convert_to('K')
    converted = temp.convert_to('C')

    assert converted == temp

def test_comparisons():
    assert Temperature(0, 'C') == Temperature(32, 'F')
    assert Temperature(0, 'C') < Temperature(100, 'C')
    assert Temperature(100, 'C') > Temperature(32, 'F')
    assert Temperature(0, 'C') <= Temperature(32, 'F')
    assert Temperature(100, 'C') >= Temperature(212, 'F')
    assert Temperature(0, 'C') != Temperature(100, 'C')

def test_str():
    temp = Temperature(25, 'C')
    assert str(temp) == '25.00 °C'

def test_repr():
    temp = Temperature(25, 'C')
    assert repr(temp) == 'Temperature(25, C)'

============================= test session starts ==============================
platform darwin -- Python 3.14.10, pytest-9.2.0, pluggy-1.7.0 -- /Users/tommy/Documents/Skole/INF201-Deliverables/Week_48/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/tommy/Documents/Skole/INF201-Deliverables/Week_48
plugins: platformdirs-4.14.8, anyio-4.16.0
collected 8 items                                                              

test_temperature.py::test_initialization PASSED                          [ 12%]
test_temperature.py::test_read_only_properties PASSED                    [ 25%]
test_temperature.py::test_validation PASSED                              [ 37%]
test_temperature.py::test_conversions PASSED                             [ 50%]
test_temperature.py::test_roundtrip PASSED                               [ 62%]
test_temperature.py::test_comparisons PASSED                             [ 75%]
test_temperature.py::test_str PASSED                                     [ 87%]
test_t

---

## Test Results

The `Temperature` class was tested using **pytest**. All 8 tests passed successfully.

| Test | Result |
|---|---|
| Initialization | <span style="color:green">✓ PASSED</span> |
| Read-only properties | <span style="color:green">✓ PASSED</span> |
| Validation | <span style="color:green">✓ PASSED</span> |
| Temperature conversions | <span style="color:green">✓ PASSED</span> |
| Round-trip conversions | <span style="color:green">✓ PASSED</span> |
| Comparisons | <span style="color:green">✓ PASSED</span> |
| String formatting (`str`) | <span style="color:green">✓ PASSED</span> |
| Representation formatting (`repr`) | <span style="color:green">✓ PASSED</span> |

**Summary:** 8 passed in 0.09 seconds.

All tests passed, confirming that the class behaves as expected for the tested functionality.

---

## Task 2: *Fractions*

For Task 2, we are writing a `Fraction` class that represents fractions. Both the numerator and the denominator are integers, and the denominator is always strictly positive (>0). The class is written in its own script, `fraction.py`, so it can be imported and tested with `pytest` in `test_fraction.py`.

The work is split in two parts:

1. **2.1** - *The initialiser and all six comparisons (`==`, `!=`, `<`, `<=`, `>`, `>=`).*
2. **2.2** - *Four other special methods for arithmetic (`+`, `-`, `*`, `/`).*

Before splitting the work, we agreed on a few things so that the two parts fit together:

* The numerator and denominator are stored as the read-only attributes `numerator` and `denominator`.
* A fraction is always stored in lowest terms, so `Fraction(2, 4)` becomes `1/2`.
* The sign is always kept in the numerator, so `Fraction(1, -2)` becomes `-1/2` and the denominator stays positive.

This means the arithmetic methods in 2.2 can simply create a new `Fraction` from the result, and the initialiser makes sure it is valid and simplified.

---

### 2.1 *Initialiser and comparisons*

The initialiser takes a numerator and a denominator (default 1), and makes sure the fraction is valid and simplified, step by step:

1. `__init__` - *Checks that both values are integers (`bool` is not allowed, even though it is a subclass of `int`), and raises a `TypeError` if not. Raises a `ZeroDivisionError` if the denominator is 0. If the denominator is negative, the sign of both is flipped so the sign ends up in the numerator. Finally, both are divided by their greatest common divisor (`math.gcd`), so the fraction is stored in lowest terms.*
2. `numerator` / `denominator` - *Read-only properties, so a fraction cannot be changed after it is created.*
3. `__eq__` / `__ne__` - *Since fractions are always stored in lowest terms, two fractions are equal exactly when their numerators and denominators are equal.*
4. `__lt__` / `__le__` / `__gt__` / `__ge__` - *Compared by cross-multiplication: $\frac{a}{b} < \frac{c}{d} \iff a \cdot d < c \cdot b$. This works because both denominators are always positive, so the inequality does not flip.*

All six comparisons return `NotImplemented` if the other object is not a `Fraction`. Python then gives `False` for `==`, `True` for `!=` and a `TypeError` for `<`, `<=`, `>` and `>=`.

In [5]:
class Fraction:
    """A fraction with an integer numerator and a strictly positive integer denominator."""

    def __init__(self, numerator: int, denominator: int = 1) -> None:
        """
        Create a fraction in lowest terms, with the sign kept in the numerator.

        Args:
            numerator (int): The numerator.
            denominator (int): The denominator, cannot be 0. Defaults to 1.

        Raises:
            TypeError: If numerator or denominator is not an int.
            ZeroDivisionError: If denominator is 0.
        """
        # type() instead of isinstance(), so that True/False are not accepted
        if type(numerator) is not int or type(denominator) is not int:
            raise TypeError("Numerator and denominator must be integers.")
        if denominator == 0:
            raise ZeroDivisionError("Denominator cannot be 0.")

        # Keep the sign in the numerator, so the denominator is always > 0
        if denominator < 0:
            numerator, denominator = -numerator, -denominator

        # Store the fraction in lowest terms
        divisor = math.gcd(numerator, denominator)
        self._numerator = numerator // divisor
        self._denominator = denominator // divisor

    @property
    def numerator(self) -> int:
        """The numerator (read-only)."""
        return self._numerator

    @property
    def denominator(self) -> int:
        """The denominator (read-only), always > 0."""
        return self._denominator

    def __eq__(self, other: object) -> bool:
        """Return True if both fractions have the same value."""
        if not isinstance(other, Fraction):
            return NotImplemented
        return (self.numerator == other.numerator
                and self.denominator == other.denominator)

    def __ne__(self, other: object) -> bool:
        """Return True if the fractions have different values."""
        if not isinstance(other, Fraction):
            return NotImplemented
        return not self == other

    def __lt__(self, other: object) -> bool:
        """Return True if self is smaller than other (a/b < c/d <=> a*d < c*b)."""
        if not isinstance(other, Fraction):
            return NotImplemented
        return self.numerator * other.denominator < other.numerator * self.denominator

    def __le__(self, other: object) -> bool:
        """Return True if self is smaller than or equal to other."""
        if not isinstance(other, Fraction):
            return NotImplemented
        return self.numerator * other.denominator <= other.numerator * self.denominator

    def __gt__(self, other: object) -> bool:
        """Return True if self is greater than other."""
        if not isinstance(other, Fraction):
            return NotImplemented
        return self.numerator * other.denominator > other.numerator * self.denominator

    def __ge__(self, other: object) -> bool:
        """Return True if self is greater than or equal to other."""
        if not isinstance(other, Fraction):
            return NotImplemented
        return self.numerator * other.denominator >= other.numerator * self.denominator

---

### 2.2 *Other special methods*

1. `__add__` - *Returns the sum of two fractions.* $\frac{a}{b} + \frac{c}{d} = \frac{a \cdot d + c \cdot b}{b \cdot d}$
2. `__sub__` - *Returns the difference of two fractions.* $\frac{a}{b} - \frac{c}{d} = \frac{a \cdot d - c \cdot b}{b \cdot d}$
3. `__mul__` - *Returns the product of two fractions.* $\frac{a}{b} \cdot \frac{c}{d} = \frac{a \cdot c}{b \cdot d}$
4. `__truediv__` - *Returns the quotient of two fractions.* $\frac{a}{b} \div \frac{c}{d} = \frac{a \cdot d}{b \cdot c}$

Each method returns a new `Fraction`, so `__init__` from *2.1* already reduces the result and keeps the denominator positive. The code below builds on 2.1, so its methods are inherited.

In [6]:
class Fraction(Fraction):
    """A fraction with an integer numerator and a strictly positive integer denominator."""

    def __add__(self, other: object) -> "Fraction":
        """Return the sum of self and other."""
        if not isinstance(other, Fraction):
            return NotImplemented
        return Fraction(self.numerator * other.denominator + other.numerator * self.denominator,
                        self.denominator * other.denominator)

    def __sub__(self, other: object) -> "Fraction":
        """Return the difference of self and other."""
        if not isinstance(other, Fraction):
            return NotImplemented
        return Fraction(self.numerator * other.denominator - other.numerator * self.denominator,
                        self.denominator * other.denominator)

    def __mul__(self, other: object) -> "Fraction":
        """Return the product of self and other."""
        if not isinstance(other, Fraction):
            return NotImplemented
        return Fraction(self.numerator * other.numerator,
                        self.denominator * other.denominator)

    def __truediv__(self, other: object) -> "Fraction":
        """Return the quotient of self and other."""
        if not isinstance(other, Fraction):
            return NotImplemented
        return Fraction(self.numerator * other.denominator,
                        self.denominator * other.numerator)


---

### Testing with pytest

The tests are in `test_fraction.py`, and use small hand-made examples where the result is easy to check by hand. For 2.1, the tests check that:

* Fractions are stored in lowest terms, with the sign in the numerator and a default denominator of 1.
* Invalid input (non-integers, `bool` and a denominator of 0) raises the correct error.
* `numerator` and `denominator` are read-only.
* All six comparisons give the correct result, also for equal fractions written differently (`1/2` and `2/4`), for negative fractions, and when sorting a list of fractions.
* Comparing with something that is not a `Fraction` gives `False` for `==` and a `TypeError` for `<`.

For 2.2, the tests check that:

* `+`, `-`, `*` and `/` give the correct result for simple fractions, e.g. $\frac{1}{2} + \frac{1}{3} = \frac{5}{6}$ and $\frac{1}{2} \div \frac{1}{3} = \frac{3}{2}$.
* The arithmetic also works with negative fractions, and gives negative results where it should, e.g. $\frac{1}{2} - \frac{3}{4} = -\frac{1}{4}$.
* Adding a fraction and its negative gives zero ($\frac{1}{2} + (-\frac{1}{2}) = \frac{0}{1}$).
* Dividing by a fraction equal to 0 raises a `ZeroDivisionError`.
* Using `+`, `-`, `*` or `/` with something that is not a `Fraction` raises a `TypeError`.



In [7]:
import sys
!"{sys.executable}" -m pytest -v test_fraction.py

============================= test session starts ==============================
platform darwin -- Python 3.14.10, pytest-9.2.0, pluggy-1.7.0 -- /Users/tommy/Documents/Skole/INF201-Deliverables/Week_48/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/tommy/Documents/Skole/INF201-Deliverables/Week_48
plugins: platformdirs-4.14.8, anyio-4.16.0
collected 25 items                                                             

test_fraction.py::test_init_lowest_terms_and_sign[2-4-expected0] PASSED  [  4%]
test_fraction.py::test_init_lowest_terms_and_sign[3-1-expected1] PASSED  [  8%]
test_fraction.py::test_init_lowest_terms_and_sign[1--2-expected2] PASSED [ 12%]
test_fraction.py::test_init_lowest_terms_and_sign[-2--4-expected3] PASSED [ 16%]
test_fraction.py::test_init_lowest_terms_and_sign[0-5-expected4] PASSED  [ 20%]
test_fraction.py::test_init_default_denominator PASSED                   [ 24%]
test_fraction.py::test_zero_denominator_raises PASSED                    [ 28%]
test_

---

### Results and discussion of Task 2

**2.1:** All 17 tests for the initialiser and the comparisons passed. Fractions are stored in lowest terms with the sign in the numerator, invalid input gives the expected errors, and the comparisons give the correct result both for positive and negative fractions and for equal fractions written in different ways.

One choice we made is that a negative denominator is allowed as *input*, but is moved to the numerator, instead of raising an error. The assignment says the denominator should be strictly positive, and this way it always is inside the object. We also chose not to accept `bool`, even though `True` and `False` technically are integers in Python, since `Fraction(True, 2)` is most likely a mistake.

**2.2:** All 8 tests for the arithmetic operations pass. Using the methods correctly will give the right answer, and the appropriate errors are raised when a method is used with something that is not a `Fraction`, or when dividing by a fraction equal to zero.

Negative results are also handeled correctly.

---

# Use of Generative AI

### Task 1

Asked AI about how to structure the comparison methods and to help with the pytest file.

Asked for help with structuring the results markdown

---

### Task 2
#### Task 2.1

**Questions about the code**
* Asked Claude for a second opinion on how to best structure the `Fraction` class into two independent parts (2.1 and 2.2) so my group partner and I could work on them simultaneously without merge conflicts.

* Double-checked the project structure requirements for `pytest` and the notebook hand-in, confirming that placing the class in a separate `fraction.py` module allows clean importing into both `test_fraction.py` and the Jupyter notebook.

**Code written by AI**
* Used Claude to generate boilerplate templates for the six comparison methods: `__eq__` and `__ne__` compare numerator and denominator directly (fractions are always in lowest terms), while `__lt__`, `__le__`, `__gt__` and `__ge__` use cross-multiplication.

* Asked for feedback on phrasing and grammar for parts of the Task 2 introduction and 2.1 explanation text that I had drafted.

**Testing and discussion**
* Ran `pytest` on `test_fraction.py` locally, verifying that all 17 tests for 2.1 passed, and then executed the test suite inside the notebook to capture the output for the report.

**Errors and corrections**
* Initially forgot that `bool` is a subclass of `int` in Python, which caused `isinstance(True, int)` to pass during input validation. After checking with Claude on the cleanest way to handle this, I updated `__init__` to explicitly reject booleans, and standardized moving negative signs from the denominator to the numerator.

**What I did myself**
* Agreed with my group partner on the task split and attribute naming conventions, implemented the core `Fraction` class (`__init__` with type/zero validation, `math.gcd` reduction, and read-only `@property` getters), wrote the primary unit tests, assembled the notebook cells, and ran the full test suite.


#### Task 2.2

**Questions about the code**
* Asked Claude which four special methods would make sense for a `Fraction` class, and chose `__add__`, `__sub__`, `__mul__` and `__truediv__`, since they are natural for fractions and easy to check by hand.

* Asked Claude to explain how the arithmetic formulas are written in code, since there is no division in them. Learned that `Fraction(top, bottom)` stores the two numbers as a fraction, and that the initialiser from 2.1 reduces the result, so the methods only need `*`, `+` and `-`.

**Code written by AI**
* Claude gave an example of the `@pytest.mark.parametrize` style already used in `test_fraction.py`, and explained that a decorator only applies to the function directly below it.

* Asked Claude for help writing the formulas in the notebook with LaTeX, and for the description of what the 2.2 tests check.

**Testing and discussion**
* Asked Claude which cases are worth testing, and wrote tests for correct results, negative results, a sum of zero, dividing by zero and using a non-`Fraction`. All 25 tests passed, and the output is included above.

**Errors and corrections**
* The division method was first named `__fracdiv__`, so `/` raised a `TypeError`. Claude explained that Python only looks for fixed names, and `/` always calls `__truediv__`.

* Three of my test functions did not start with `test_`, so pytest silently skipped them. After renaming them, all 8 tests for 2.2 ran.

* In the notebook, the 2.2 cell started with `class Fraction:`, which replaced the 2.1 class and removed its initialiser. Changing it to `class Fraction(Fraction):` makes it inherit everything from 2.1.

**What I did myself**
* Implemented the four arithmetic methods in `fraction.py`, wrote the tests for 2.2 in `test_fraction.py`, added the 2.2 cells to the notebook and ran the test suite.


---